## CoAtNet

### Review:

* Brings in the best of both worlds. CNNS are good at local geatures and have nice inductive biases. Transformers are good at global relationships. If you stack CNNs and transformer blocks, you get good results. 
* **Relative Attention**. Take away the absolute positional embeddings like we had in the vanilla ViT, and instead use relative position biases in attentnions scores. This generalizes nicely to different input sizes. 

In [2]:
from torchvision.datasets import CIFAR100
import torchvision.transforms as transforms
import torch
import torch.nn as nn
import torch.nn.functional as F # This gives us the softmax()
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.Resize((56, 56)),
    transforms.ToTensor(),
    transforms.Normalize((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761)),
])

train_dataset = CIFAR100(root='./data', train=True, download=True, transform=transform)
test_dataset = CIFAR100(root='./data', train=False, download=True, transform=transform)

In [3]:
from vit import ViT, PatchEmbedding, MultiHeadAttention, TransformerEncoder ,ClassificationHead , FeedForward                                                               

We basically need to replace the PatchEmbedding with a deeper CNN

In [ ]:
class CNNStem(nn.Module):
    def __init__(self, num_hiddens):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size =3, stride = 2, padding =1),
            nn.BatchNorm2d(64),
            nn.GELU(),
            nn.Conv2d(64, 128, kernel_size =3, stride = 2, padding =1),
            nn.BatchNorm2d(128),
            nn.GELU(),
            nn.Conv2d(128, num_hiddens, kernel_size =3, stride = 2, padding =1),
            nn.BatchNorm2d(num_hiddens),
            nn.GELU()
        )
    def forward(self, x):
        x = self.stem(x)
        return x
     

In [25]:
img = torch.randn(1, 3, 56, 56)

In [8]:
stem = CNNStem(num_hiddens = 512)
outs = stem(x)
print(outs.shape)

torch.Size([2, 512, 7, 7])


In [40]:
class Vit(nn.Module):
    def __init__(self, img_size, patch_size, num_hiddens, num_heads, num_classes, depth=6, dropout=0.1):
        super().__init__()
        self.num_patches = (img_size // patch_size) ** 2
        self.patch_embedding = PatchEmbedding(img_size, patch_size, num_hiddens)
        self.cls_token = nn.Parameter(torch.randn(1, 1, num_hiddens))
        self.pos_embedding = nn.Parameter(torch.randn(1, self.num_patches + 1, num_hiddens))
        self.encoder = TransformerEncoder(num_hiddens, depth, num_heads, mlp_dim=num_hiddens * 4)
        self.classification_head = ClassificationHead(num_hiddens, num_classes)

    def forward(self, x):
        batch = x.shape[0]
        patches = self.patch_embedding(x)
        print(patches.shape)
        patches = patches.flatten(2).transpose(1, 2)
        print(patches.shape)
        cls_tokens = self.cls_token.expand(batch, -1, -1)
        x = torch.cat((cls_tokens, patches), dim=1)
        x = x + self.pos_embedding
        x = self.encoder(x)
        x = self.classification_head(x)
        return x
    
class CoAtNEt(nn.Module):
    def __init__(self, img_size, patch_size, num_hiddens, num_heads, num_classes, depth=6, dropout=0.1):
        super().__init__()
        self.num_patches = 7*7
        self.stem = CNNStem(num_hiddens)
        self.cls_token = nn.Parameter(torch.randn(1, 1, num_hiddens))
        self.pos_embedding = nn.Parameter(torch.randn(1, self.num_patches + 1, num_hiddens))
        self.encoder = TransformerEncoder(num_hiddens, depth, num_heads, mlp_dim=num_hiddens * 4)
        self.classification_head = ClassificationHead(num_hiddens, num_classes)
        self.patch_size = patch_size

    def forward(self, x):
        batch = x.shape[0]
        patches_equivalent = self.stem(x)
        patches_equivalent = patches_equivalent.flatten(2).transpose(1, 2)
        cls_tokens = self.cls_token.expand(batch, -1, -1)
        x = torch.cat((cls_tokens, patches_equivalent), dim=1)
        x = x + self.pos_embedding
        x = self.encoder(x)
        x = self.classification_head(x)
        return x

In [41]:
net = Vit(patch_size = 7, num_hiddens=512, num_heads = 8, num_classes= 100, img_size = 56)
coatnet = CoAtNEt(patch_size = 7, num_hiddens=512, num_heads = 8, num_classes= 100, img_size = 56)

In [38]:
x = net(img)

torch.Size([1, 512, 8, 8])
torch.Size([1, 64, 512])


In [39]:
x = coatnet(img)

torch.Size([1, 512, 7, 7])
torch.Size([1, 49, 512])


In [42]:
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Using: {device}")

Using: mps


In [43]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [47]:
coatnet.to(device)

CoAtNEt(
  (stem): CNNStem(
    (stem): Sequential(
      (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): GELU(approximate='none')
      (3): Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (4): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (5): GELU(approximate='none')
      (6): Conv2d(128, 512, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (7): BatchNorm2d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (8): GELU(approximate='none')
    )
  )
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-5): 6 x TransformerBlock(
        (norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (attn): MultiHeadAttention(
          (heads): ModuleList(
            (0-7): 8 x AttentionHead(
              (q): Linear(in_features=512, out_featu

In [48]:
import time
optimizer = torch.optim.Adam(coatnet.parameters(), lr=3e-4)
criterion = nn.CrossEntropyLoss()

for epoch in range(50):
    coatnet.train()
    total_loss, correct, total = 0, 0, 0
    start = time.time()

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        logits = coatnet(images)
        loss = criterion(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        correct += (logits.argmax(dim=-1) == labels).sum().item()
        total += labels.size(0)
    elapsed = time.time() - start
    print(f"Epoch {epoch+1} | Loss: {total_loss/len(train_loader):.4f} | Acc: {correct/total:.4f} | Time: {elapsed:.1f}s")

Epoch 1 | Loss: 3.6707 | Acc: 0.1298 | Time: 234.9s
Epoch 2 | Loss: 3.1504 | Acc: 0.2177 | Time: 224.4s
Epoch 3 | Loss: 2.9129 | Acc: 0.2650 | Time: 223.2s
Epoch 4 | Loss: 2.7719 | Acc: 0.2941 | Time: 220.1s
Epoch 5 | Loss: 2.6528 | Acc: 0.3191 | Time: 220.2s
Epoch 6 | Loss: 2.5491 | Acc: 0.3434 | Time: 220.4s
Epoch 7 | Loss: 2.4791 | Acc: 0.3539 | Time: 220.2s
Epoch 8 | Loss: 2.3921 | Acc: 0.3714 | Time: 220.2s
Epoch 9 | Loss: 2.3086 | Acc: 0.3898 | Time: 221.1s
Epoch 10 | Loss: 2.2702 | Acc: 0.3980 | Time: 221.0s
Epoch 11 | Loss: 2.2146 | Acc: 0.4096 | Time: 220.9s
Epoch 12 | Loss: 2.1748 | Acc: 0.4175 | Time: 222.1s
Epoch 13 | Loss: 2.1202 | Acc: 0.4317 | Time: 220.7s
Epoch 14 | Loss: 2.0260 | Acc: 0.4503 | Time: 221.0s
Epoch 15 | Loss: 1.9948 | Acc: 0.4582 | Time: 220.7s
Epoch 16 | Loss: 1.9251 | Acc: 0.4743 | Time: 221.8s
Epoch 17 | Loss: 1.8899 | Acc: 0.4811 | Time: 221.2s
Epoch 18 | Loss: 1.7947 | Acc: 0.5026 | Time: 220.7s
Epoch 19 | Loss: 1.7059 | Acc: 0.5215 | Time: 220.5s
Ep

## Next Steps



Above we have: CNN Stem → Transformer Encoder (all attention blocks)                                                                                                                              
                                                                                                                                                                                    
The actual CoAtNet paper stacks them in stages:                                                                                                                                    
                
Stage 0: Conv blocks (S0)
Stage 1: Conv blocks (S1)
Stage 2: Transformer blocks (S2)
Stage 3: Transformer blocks (S3)

In [53]:
# !pip install timm

In [56]:
import timm                                                                                                                                                                        
                                                                                                                                                                                     
# List available coat models                                                                                                                                                       
print(timm.list_models('coatnet*'))                                                                                                                                                   
                                                                                                                                                                                    
# Load one and inspect
model = timm.create_model('coatnet_0_224', pretrained=False)
print(model)

['coatnet_0_224', 'coatnet_0_rw_224', 'coatnet_1_224', 'coatnet_1_rw_224', 'coatnet_2_224', 'coatnet_2_rw_224', 'coatnet_3_224', 'coatnet_3_rw_224', 'coatnet_4_224', 'coatnet_5_224', 'coatnet_bn_0_rw_224', 'coatnet_nano_cc_224', 'coatnet_nano_rw_224', 'coatnet_pico_rw_224', 'coatnet_rmlp_0_rw_224', 'coatnet_rmlp_1_rw2_224', 'coatnet_rmlp_1_rw_224', 'coatnet_rmlp_2_rw_224', 'coatnet_rmlp_2_rw_384', 'coatnet_rmlp_3_rw_224', 'coatnet_rmlp_nano_rw_224']
MaxxVit(
  (stem): Stem(
    (conv1): Conv2d(3, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (norm1): BatchNormAct2d(
      64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
      (drop): Identity()
      (act): GELU()
    )
    (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  )
  (stages): Sequential(
    (0): MaxxVitStage(
      (blocks): Sequential(
        (0): MbConvBlock(
          (shortcut): Downsample2d(
            (pool): AvgPool2d(kernel_size=2, s

### Undersanding the timm implemetation


#### Stem

* Conv2d(3→64, stride=2) → BatchNorm → GELU → Conv2d(64→64)

#### Stage 0 + Stage 1: MbConvBlocks (CNN stages)

* here, timm uses depth-wise comvilutions like the ones introduces in mobilenetv3. 

* What was a depthwise conv? each channel is convolved independently for speed. 

#### Stage 2 + Stage 3: TransformerBlock2d (Attention stages)

* this is very much like our ViT implementations with some tweaks that we should look into.

* Timm uses relative position bias, whears we were using absolute learned embeddings.

* No cls_token

* This one is interesting, instead of using nn.Linear in the Attention, they use a conv2:

```python
class AttentionHead(nn.Module):
    def __init__(self, dim, head_dim, dropout=0.1):
        super().__init__()
        self.scale = head_dim**-0.5
        self.q = nn.Linear(dim, head_dim, bias=False)
        self.k = nn.Linear(dim, head_dim, bias=False)
        self.v = nn.Linear(dim, head_dim, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        q, k, v = self.q(x), self.k(x), self.v(x)
        attn = torch.matmul(q, k.transpose(-1, -2)) * self.scale
        attn = F.softmax(attn, dim=-1)
        attn = self.dropout(attn)
        out = torch.matmul(attn, v)
        return out
```

So instead of doing:

```python
self.q = nn.Linear(dim, head_dim, bias=False)
self.k = nn.Linear(dim, head_dim bias=False)
self.v = nn.Linear(dim, head_dim, bias=False)
```

they do:

```python
self.qkv = nn.Conv2d(dim, dim * 3, kernel_size=1)  # Q, K, V all in one
```